# freqk_gr — Explore Allele Frequency Results

Reads in all completed `allele_frequencies.k31.tsv` files from `results/`,  
computes ALT allele frequency per variant per sample, and does basic EDA.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path

# ── paths ──────────────────────────────────────────────────────────────────
RESULTS  = Path("/home/tbellagio/scratch/freqk_gr/results")
DATA     = Path("/home/tbellagio/scratch/freqk_gr/data")
K = 31

# NOTE: we use counts_by_allele (ref_kmers|alt_kmers) rather than
# allele_frequencies. freqk call outputs binary 0/1 majority-vote calls,
# not continuous frequencies. AF = alt / (ref + alt).
#
# Flower counts (N = flowerscollected) are used to:
#   - set minimum detectable AF = 1/N per sample
#   - estimate allele count k = round(AF * N)  [valid for selfing Arabidopsis]
#   - compute uncertainty: sd ~ sqrt(AF*(1-AF)/N)

## 1. Discover completed samples

In [ ]:
af_files = sorted(RESULTS.glob(f"*/*/k{K}/*.counts_by_allele.k{K}.tsv"))
print(f"Completed samples: {len(af_files)}")
for f in af_files:
    print(" ", f.parent.parent.name)

## 2. Load all results into a single DataFrame

In [ ]:
def parse_sample_id(sample_id):
    """Extract site, plot, date, year from MLFH{site}{plot}{date}."""
    site  = sample_id[4:6]
    plot  = sample_id[6:8]
    date  = sample_id[8:16]
    year  = sample_id[8:12]
    return site, plot, date, year

def load_af(path, k=K):
    """Load counts_by_allele file, compute AF = alt / (ref + alt)."""
    sample_id = path.parent.parent.name
    site, plot, date, year = parse_sample_id(sample_id)

    df = pd.read_csv(path, sep="|", header=None, names=["ref", "alt"], dtype=float)
    df[["ref", "alt"]] = df[["ref", "alt"]].fillna(0).astype(int)

    df["sample_id"]   = sample_id
    df["site"]        = site
    df["plot"]        = plot
    df["date"]        = pd.to_datetime(date, format="%Y%m%d")
    df["year"]        = year
    df["variant_idx"] = df.index  # 0-based row = variant index

    total = df["ref"] + df["alt"]
    df["af"] = np.where(total > 0, df["alt"] / total, np.nan)
    return df

dfs = [load_af(f) for f in af_files]
df = pd.concat(dfs, ignore_index=True)
print(df.shape)
df.head()

## 3. Join sample metadata (flower counts, generation, coverage)

In [ ]:
# Load sample metadata
meta = pd.read_csv(DATA / "samples_data_fix57.csv", usecols=[
    "sampleid", "flowerscollected", "generation", "fix_57_generation",
    "coverage", "weighted_mean_coverage", "usesample"
]).rename(columns={"sampleid": "sample_id"})

# Join onto main df
df = df.merge(meta, on="sample_id", how="left")

# Add min detectable AF and allele count estimate
df["N"]       = df["flowerscollected"]
df["min_af"]  = 1 / df["N"]
df["af_se"]   = np.sqrt(df["af"] * (1 - df["af"]) / df["N"])  # binomial SE

# recompute covered with metadata
df["coverage_kmers"] = df["ref"] + df["alt"]
covered = df[df["af"].notna() & (df["coverage_kmers"] > 0)]

print(f"Samples with metadata: {df['N'].notna().sum()} / {len(df)} rows")
print(f"\nFlower counts for site 04:")
print(df[df["site"]=="04"].groupby(["sample_id","N","generation"])["af"].count().reset_index(
    ).drop(columns="af").drop_duplicates().to_string(index=False))

## 4. Basic QC

In [ ]:
# variants with zero k-mer coverage → uncovered
n_uncovered = df.groupby("sample_id").apply(lambda x: (x["coverage_kmers"] == 0).sum())
n_total     = df.groupby("sample_id").size()
n_below_min = covered.groupby("sample_id").apply(lambda x: (x["af"] < x["min_af"]).sum())

qc = pd.DataFrame({"total": n_total, "uncovered": n_uncovered, "below_min_af": n_below_min})
qc["pct_uncovered"]   = 100 * qc["uncovered"]   / qc["total"]
qc["pct_below_min_af"] = 100 * qc["below_min_af"] / qc["total"]
qc = qc.sort_values("pct_uncovered")
print(qc.to_string())

In [ ]:
# AF distribution per sample — only covered variants
fig, ax = plt.subplots(figsize=(10, 4))
for sid, grp in covered.groupby("sample_id"):
    grp["af"].plot.hist(bins=50, alpha=0.4, label=sid, ax=ax, density=True)
ax.set_xlabel("ALT allele frequency")
ax.set_ylabel("Density")
ax.set_title("AF distribution per sample")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

covered["coverage_kmers"].clip(upper=200).plot.hist(
    bins=50, ax=axes[0], color="steelblue", edgecolor="white"
)
axes[0].set_xlabel("Total k-mer coverage (ref + alt, clipped at 200)")
axes[0].set_ylabel("Variants")
axes[0].set_title("K-mer coverage distribution (all samples)")

med_cov = covered.groupby("sample_id")["coverage_kmers"].median().sort_values()
med_cov.plot.barh(ax=axes[1], color="steelblue")
axes[1].set_xlabel("Median k-mer coverage")
axes[1].set_title("Median k-mer coverage per sample")

plt.tight_layout()
plt.show()

## 5. AF over time (site 04)

In [ ]:
# mean ALT AF per time point across plots — site 04 only
site04 = covered[covered["site"] == "04"].copy()

# date → generation mapping
date_gen = (
    site04.drop_duplicates("date")[["date", "generation"]]
    .set_index("date")["generation"]
    .to_dict()
)

time_af = (
    site04
    .groupby(["variant_idx", "date"])["af"]
    .mean()
    .reset_index()
)

# summarise across variants: mean ± std
summary = time_af.groupby("date")["af"].agg(["mean", "std"]).reset_index()
summary["generation"] = summary["date"].map(date_gen)

gen_colors = {1: "steelblue", 2: "darkorange", 3: "forestgreen"}

fig, ax = plt.subplots(figsize=(8, 4))
for gen, grp in summary.groupby("generation"):
    ax.errorbar(grp["date"], grp["mean"], yerr=grp["std"],
                fmt="o-", capsize=4, color=gen_colors.get(gen, "gray"),
                label=f"Gen {int(gen)}")
ax.set_xlabel("Date")
ax.set_ylabel("Mean ALT allele frequency")
ax.set_title("Site 04 — mean AF across variants over time")
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%Y-%m-%d"))
ax.legend(title="Generation")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


## 6. Per-plot AF comparison

In [ ]:
# Add generation label to x-axis tick labels
site04["date_gen"] = site04.apply(
    lambda r: (
        f"{r['date'].strftime('%Y-%m-%d')}\n(Gen {int(r['generation'])})"
        if pd.notna(r["generation"]) else r["date"].strftime("%Y-%m-%d")
    ),
    axis=1
)

fig, ax = plt.subplots(figsize=(11, 4))
sns.boxplot(
    data=site04,
    x="date_gen", y="af", hue="plot",
    ax=ax, flierprops=dict(marker=".", alpha=0.2, markersize=1)
)
ax.set_xlabel("Date (Generation)")
ax.set_ylabel("ALT allele frequency")
ax.set_title("Site 04 — AF per plot per time point")
ax.legend(title="Plot", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()


## 7. Seed mix (founder population baseline)

In [ ]:
# Load seed mix counts_by_allele files
sm_files = sorted(RESULTS.glob(f"seedmix/*/k{K}/*.counts_by_allele.k{K}.tsv"))
print(f"Seed mix samples found: {len(sm_files)}")
for f in sm_files:
    print(" ", f.parent.parent.name)

def load_seedmix(path, k=K):
    """Load a seed mix counts_by_allele file."""
    sample_id = path.parent.parent.name  # e.g. SEEDMIX_S1
    df = pd.read_csv(path, sep="|", header=None, names=["ref", "alt"], dtype=float)
    df[["ref", "alt"]] = df[["ref", "alt"]].fillna(0).astype(int)
    df["sample_id"]   = sample_id
    df["variant_idx"] = df.index
    total = df["ref"] + df["alt"]
    df["af"] = np.where(total > 0, df["alt"] / total, np.nan)
    df["coverage_kmers"] = total
    return df

if sm_files:
    df_sm = pd.concat([load_seedmix(f) for f in sm_files], ignore_index=True)
    covered_sm = df_sm[df_sm["af"].notna() & (df_sm["coverage_kmers"] > 0)]
    print(f"\nSeed mix shape: {df_sm.shape}")
    df_sm.head()
else:
    print("No seed mix results yet — submit jobs with: python scripts/launch_seedmix.py")


In [ ]:
# Compare seed mix mean AF vs site 04 mean AF per variant
# (only variants covered in both)
if sm_files and len(sm_files) > 0:
    # Mean AF across seed mix replicates per variant
    sm_mean = (
        covered_sm.groupby("variant_idx")["af"]
        .mean()
        .rename("af_seedmix")
    )

    # Mean AF per variant per date for site 04
    site04_mean = (
        site04.groupby(["variant_idx", "date", "generation"])["af"]
        .mean()
        .reset_index()
        .rename(columns={"af": "af_field"})
    )

    merged = site04_mean.join(sm_mean, on="variant_idx")
    merged["delta_af"] = merged["af_field"] - merged["af_seedmix"]

    # Plot: distribution of delta_af per generation
    gen_colors = {1: "steelblue", 2: "darkorange", 3: "forestgreen"}
    fig, ax = plt.subplots(figsize=(8, 4))
    for gen, grp in merged.groupby("generation"):
        grp["delta_af"].dropna().plot.hist(
            bins=60, alpha=0.5, density=True,
            label=f"Gen {int(gen)}",
            color=gen_colors.get(gen, "gray"),
            ax=ax
        )
    ax.axvline(0, color="black", lw=1, ls="--")
    ax.set_xlabel("ΔALT frequency  (field − seed mix)")
    ax.set_ylabel("Density")
    ax.set_title("Site 04 — shift in ALT AF relative to seed mix")
    ax.legend(title="Generation")
    plt.tight_layout()
    plt.show()
else:
    print("Run seed mix jobs first.")


## 8. Filter to variants that changed vs seed mix

Seed mix = founder population, 8 replicates. Assume infinite N
(pool of many thousands of seeds) → just take the mean AF across S1–S8
as the baseline.

A variant is kept for downstream analysis only if its AF in at least one
site 04 sample differs from the seed-mix baseline by more than the
threshold. Variants that stayed flat carry no signal of selection.

In [ ]:
# ── Seed mix baseline: mean AF across 8 replicates ────────────────────────
MIN_SM_REPS = 4   # variant must be covered in ≥ this many seed mix reps

sm_baseline = (
    covered_sm.groupby("variant_idx")
    .agg(
        af_seedmix     = ("af", "mean"),
        af_seedmix_std = ("af", "std"),
        n_reps_covered = ("af", "count"),
        ref_sm         = ("ref", "sum"),
        alt_sm         = ("alt", "sum"),
    )
    .reset_index()
)

sm_baseline = sm_baseline[sm_baseline["n_reps_covered"] >= MIN_SM_REPS]
print(f"Seed mix baseline variants (covered in ≥{MIN_SM_REPS}/8 reps): {len(sm_baseline):,}")
print(f"Mean across-replicate std of AF: {sm_baseline['af_seedmix_std'].mean():.4f}")
sm_baseline.head()


In [ ]:
# ── Join site 04 samples to baseline and compute ΔAF ──────────────────────
site04_vs_sm = site04.merge(sm_baseline, on="variant_idx", how="inner")
site04_vs_sm["delta_af"]  = site04_vs_sm["af"] - site04_vs_sm["af_seedmix"]
site04_vs_sm["abs_delta"] = site04_vs_sm["delta_af"].abs()

# ── Filter to variants that changed in ≥1 sample ──────────────────────────
# Threshold options:
#   (a) fixed biological effect size
#   (b) sample-specific noise floor: max(1/N, 0.05) — 1/N is the min
#       detectable AF for N flowers collected
DELTA_MIN_FIXED = 0.05
USE_NOISE_FLOOR = True   # set False for fixed threshold only

if USE_NOISE_FLOOR:
    site04_vs_sm["delta_threshold"] = np.maximum(
        site04_vs_sm["min_af"].fillna(DELTA_MIN_FIXED), DELTA_MIN_FIXED
    )
else:
    site04_vs_sm["delta_threshold"] = DELTA_MIN_FIXED

site04_vs_sm["is_changed"] = site04_vs_sm["abs_delta"] > site04_vs_sm["delta_threshold"]

changed_variants = (
    site04_vs_sm.groupby("variant_idx")["is_changed"].any()
)
changed_variants = changed_variants[changed_variants].index

print(f"Total baseline variants tested:        {site04_vs_sm['variant_idx'].nunique():,}")
print(f"Variants changed in ≥1 site 04 sample: {len(changed_variants):,}")
print(f"Fraction changed: {len(changed_variants) / site04_vs_sm['variant_idx'].nunique():.2%}")

site04_changed = site04_vs_sm[site04_vs_sm["variant_idx"].isin(changed_variants)].copy()


In [ ]:
# ── Visualise: ΔAF distribution per generation, filtered vs all ──────────
gen_colors = {1: "steelblue", 2: "darkorange", 3: "forestgreen"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, (df_plot, title) in zip(
    axes,
    [(site04_vs_sm,  "All variants"),
     (site04_changed, f"Filtered: changed in ≥1 sample")],
):
    for gen, grp in df_plot.groupby("generation"):
        grp["delta_af"].dropna().plot.hist(
            bins=80, alpha=0.5, density=True, ax=ax,
            color=gen_colors.get(gen, "gray"),
            label=f"Gen {int(gen)}",
        )
    ax.axvline(0, color="black", lw=1, ls="--")
    ax.set_xlabel("ΔALT frequency (field − seed mix)")
    ax.set_title(title)
    ax.legend(title="Generation")
axes[0].set_ylabel("Density")
plt.tight_layout()
plt.show()


In [ ]:
# ── Save the changed-variant table for downstream analyses ────────────────
out_path = DATA.parent / "results" / "site04_changed_variants.tsv"
out_path.parent.mkdir(exist_ok=True)
site04_changed[[
    "variant_idx", "sample_id", "plot", "date", "generation", "N",
    "ref", "alt", "af", "af_seedmix", "af_seedmix_std",
    "delta_af", "abs_delta", "delta_threshold", "is_changed",
]].to_csv(out_path, sep="\t", index=False)
print(f"Saved {len(site04_changed):,} rows ({site04_changed['variant_idx'].nunique():,} variants) → {out_path}")
